pandas                    2.3.3


In [5]:
PATH = "/home/yashbaviskar/Desktop/Projects/india-crop-pipeline/dataset/Historical_Data/parquet/2001.paraquet"

In [1]:
import pandas as pd
import glob
import os

# Your path (using *.par*quet to catch both .parquet and .paraquet extensions)
path_pattern = "/home/yashbaviskar/Desktop/Projects/india-crop-pipeline/dataset/Historical_Data/parquet/20*.par*quet"
file_paths = sorted(glob.glob(path_pattern))

print(f"Found {len(file_paths)} files.")

# Using a Set to store unique (Commodity, Variety) tuples across all years
commodity_variety_pairs = set()

print("Extracting unique Commodities and Varieties...")
for file in file_paths:
    # Read ONLY the necessary columns to save RAM
    df_subset = pd.read_parquet(file, columns=['Commodity', 'Variety'])
    
    # Drop duplicates within the file first
    unique_in_file = df_subset.drop_duplicates()
    
    # Add to our global set
    for _, row in unique_in_file.iterrows():
        commodity_variety_pairs.add((row['Commodity'], row['Variety']))

# ---------------------------------------------------------
# TASK 1: List of ALL unique commodities
# ---------------------------------------------------------
all_commodities = sorted(list(set(c for c, v in commodity_variety_pairs if pd.notna(c))))
print(f"\nTotal Unique Commodities: {len(all_commodities)}")
# print(all_commodities[:20]) # Print first 20 to check

# ---------------------------------------------------------
# TASK 2: List of ALL unique varieties
# ---------------------------------------------------------
all_varieties = sorted(list(set(v for c, v in commodity_variety_pairs if pd.notna(v))))
print(f"Total Unique Varieties: {len(all_varieties)}")

# ---------------------------------------------------------
# TASK 3: Grouping each Variety under its Commodity
# ---------------------------------------------------------
commodity_to_variety = {}
for c, v in commodity_variety_pairs:
    if pd.notna(c) and pd.notna(v):
        if c not in commodity_to_variety:
            commodity_to_variety[c] = []
        commodity_to_variety[c].append(v)

# Sort the varieties for each commodity for cleaner viewing
for c in commodity_to_variety:
    commodity_to_variety[c].sort()

# Example Output: Let's see the varieties for 'Onion' and 'Apple'
print("\nVarieties of Onion:", commodity_to_variety.get('Onion', 'Not Found'))
print("Varieties of Apple:", commodity_to_variety.get('Apple', 'Not Found'))

Found 26 files.
Extracting unique Commodities and Varieties...

Total Unique Commodities: 389
Total Unique Varieties: 1534

Varieties of Onion: ['1st Sort', '2nd Sort', 'Bangalore-Samall', 'Beelary-Red', 'Bellary', 'Big', 'Bombay (U.P.)', 'Dry F.A.Q.', 'Hybrid', 'Local', 'Medium', 'Nasik', 'New Pune', 'Onion', 'Onion-Organic', 'Other', 'Pole', 'Puna', 'Pusa-Red', 'Red', 'Small', 'Small - I', 'Small No. II', 'Telagi', 'Tripur', 'White']
Varieties of Apple: ['American', 'Apple', 'Condition', 'Creamjon', 'Delicious', 'Fanny', 'French', 'Golden', 'Hajratbali', 'Kalidevi', 'Kasmir/Shimla - II', 'Kesri', 'Kullu Delicious', 'Kullu Royal Delicious', 'Maharaji', 'Other', 'Radha Krishni', 'Red Gold', 'Red June', 'Rich Red', 'Rizakwadi', 'Royal Delicious', 'Royal Mishri', 'Simla']


In [3]:
# ---------------------------------------------------------
# TASK 5: Convert mapping to DataFrame and export
# ---------------------------------------------------------

# Prepare the data: list of dictionaries
mapping_data = []
for commodity, varieties in commodity_to_variety.items():
    mapping_data.append({
        'Commodity': commodity,
        'Variety_Count': len(varieties),
        'Varieties': ", ".join(varieties) # Joins the list into a clean, comma-separated string
    })

# Create a DataFrame
df_mapping = pd.DataFrame(mapping_data)

# Sort alphabetically by Commodity
df_mapping = df_mapping.sort_values(by='Commodity').reset_index(drop=True)

# Print the top 10 to the console just to check it
print("\n--- Full Commodity to Variety Mapping (Top 10) ---")
print(df_mapping.head(10))

# ---------------------------------------------------------
# EXPORT TO CSV
# ---------------------------------------------------------
# Save it to your project folder so you have a hard copy
output_file = "/home/yashbaviskar/Desktop/Projects/india-crop-pipeline/commodity_variety_mapping.csv"
df_mapping.to_csv(output_file, index=False)

print(f"\n✅ Full list successfully saved to: {output_file}")


--- Full Commodity to Variety Mapping (Top 10) ---
             Commodity  Variety_Count  \
0             Absinthe              1   
1                Ajwan              3   
2        Alasande Gram              2   
3       Almond (Badam)              2   
4           Alsandikai              2   
5           Amaranthus              4   
6          Ambada Seed              2   
7         Ambady/Mesta              2   
8  Ambady/Mesta/Patson              2   
9     Amla (Nelli Kai)              2   

                                           Varieties  
0                                           chirayta  
1                       Ajwan, Celery-Organic, Other  
2                               Alasande Gram, Other  
3                              Almond (Badam), Other  
4                                  Alsandikai, Other  
5  Amaranth Greens-Organic, Amaranthus, Cholai Bh...  
6                                 Ambada Seed, Other  
7                 Ambadi/Mesta, Ambady/Mesta-Organic  
8

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")

# 1. Load a Representative Sample for General EDA
# (e.g., 5% of each file to make memory management easy)
sample_dfs = []
for file in file_paths:
    # Read the whole file
    df = pd.read_parquet(file)
    # Take a 5% random sample
    df_sampled = df.sample(frac=0.05, random_state=42) 
    sample_dfs.append(df_sampled)

# Combine samples into one DataFrame
eda_df = pd.concat(sample_dfs, ignore_index=True)
print(f"\nCreated a sample DataFrame with {len(eda_df):,} rows for General EDA.")

# Convert Arrival_Date to datetime objects
eda_df['Arrival_Date'] = pd.to_datetime(eda_df['Arrival_Date'])

# --- A. Data Types and Missing Values ---
print("\n--- Data Info & Missing Values ---")
print(eda_df.info())
print("\nMissing Values Percentage:")
print((eda_df.isnull().sum() / len(eda_df)) * 100)

# --- B. Statistical Summary of Prices ---
print("\n--- Price Statistics (Sampled) ---")
print(eda_df[['Min_Price', 'Max_Price', 'Modal_Price']].describe())

# --- C. Top 10 States by Trading Volume (Row Count) ---
plt.figure(figsize=(12, 6))
top_states = eda_df['State'].value_counts().head(10)
sns.barplot(x=top_states.values, y=top_states.index, palette='viridis')
plt.title('Top 10 States by Number of Price Records')
plt.xlabel('Number of Records (Sampled)')
plt.ylabel('State')
plt.show()

# --- D. Top 10 Most Traded Commodities ---
plt.figure(figsize=(12, 6))
top_commodities = eda_df['Commodity'].value_counts().head(10)
sns.barplot(x=top_commodities.values, y=top_commodities.index, palette='magma')
plt.title('Top 10 Most Frequently Recorded Commodities')
plt.xlabel('Number of Records (Sampled)')
plt.ylabel('Commodity')
plt.show()

# --- E. Price Distribution Analysis (Outlier Detection) ---
# Agricultural data often has extreme outliers (e.g., typos like 999999 for price)
plt.figure(figsize=(10, 5))
sns.boxplot(x=eda_df['Modal_Price'])
plt.title('Distribution of Modal Price (Showing Outliers)')
plt.xscale('log') # Log scale because prices vary wildly between wheat and expensive spices like saffron
plt.xlabel('Modal Price (INR/Quintal) - Log Scale')
plt.show()

# --- F. Time Series Trend for a Specific Commodity (e.g., Onion) ---
# We aggregate the monthly average Modal_Price for 'Onion'
onion_df = eda_df[eda_df['Commodity'] == 'Onion'].copy()

# Set index to date for resampling
onion_df.set_index('Arrival_Date', inplace=True)

# Calculate monthly average modal price across India
# FIX: Changed 'M' to 'ME' for Pandas 2.2+ compatibility
monthly_onion_price = onion_df['Modal_Price'].resample('ME').mean()

plt.figure(figsize=(15, 6))
monthly_onion_price.plot(color='crimson', linewidth=2)
plt.title('Average Monthly Modal Price of Onions in India (2001-2026)')
plt.xlabel('Year')
plt.ylabel('Average Modal Price (INR/Quintal)')
plt.show()
# Set index to date for resampling
onion_df.set_index('Arrival_Date', inplace=True)

# Calculate monthly average modal price across India
monthly_onion_price = onion_df['Modal_Price'].resample('M').mean()

plt.figure(figsize=(15, 6))
monthly_onion_price.plot(color='crimson', linewidth=2)
plt.title('Average Monthly Modal Price of Onions in India (2001-2026)')
plt.xlabel('Year')
plt.ylabel('Average Modal Price (INR/Quintal)')
plt.show()